# 03 — Silver → Gold (Databricks)

Reads the unified Silver Delta table and produces aggregated,  
business-ready Gold tables ready for Power BI or SQL querying.

## 1. Read Silver

In [ ]:
from pyspark.sql import functions as F

silver_path = 'abfss://olistdata@olistdatastorage1105.dfs.core.windows.net/Silver/full_orders/'
gold_base   = 'abfss://olistdata@olistdatastorage1105.dfs.core.windows.net/Gold/'

silver_df = spark.read.format('delta').load(silver_path)
print(f'Silver rows: {silver_df.count()}')
display(silver_df.limit(5))


## 2. Gold Table 1 — Revenue by Product Category

In [ ]:
revenue_by_category = (
    silver_df
    .groupBy('product_category_name_english')
    .agg(
        F.round(F.sum('payment_value'),  2).alias('total_revenue'),
        F.count('order_id')              .alias('total_orders'),
        F.round(F.avg('payment_value'),  2).alias('avg_order_value'),
        F.countDistinct('customer_id')   .alias('unique_customers'),
    )
    .orderBy(F.desc('total_revenue'))
)

display(revenue_by_category)


## 3. Gold Table 2 — Delivery Performance by Seller State

In [ ]:
delivery_performance = (
    silver_df
    .filter(col('order_status') == 'delivered')
    .groupBy('seller_state')
    .agg(
        F.round(F.avg('actual_delivery_days'),    1).alias('avg_actual_delivery_days'),
        F.round(F.avg('estimated_delivery_days'), 1).alias('avg_estimated_delivery_days'),
        F.round(
            F.avg(F.col('delay').cast('int')) * 100, 1
        ).alias('delay_rate_pct'),
        F.count('order_id').alias('total_orders'),
    )
    .orderBy(F.desc('delay_rate_pct'))
)

display(delivery_performance)


## 4. Gold Table 3 — Monthly Revenue Trend

In [ ]:
monthly_revenue = (
    silver_df
    .withColumn('year_month', F.date_format(col('order_purchase_timestamp'), 'yyyy-MM'))
    .groupBy('year_month')
    .agg(
        F.round(F.sum('payment_value'), 2).alias('total_revenue'),
        F.count('order_id')              .alias('total_orders'),
        F.countDistinct('customer_id')   .alias('unique_customers'),
    )
    .orderBy('year_month')
)

display(monthly_revenue)


## 5. Gold Table 4 — Review Score Distribution

In [ ]:
review_scores = (
    silver_df
    .groupBy('review_score')
    .agg(
        F.count('review_id')                      .alias('total_reviews'),
        F.round(F.avg('payment_value'), 2)        .alias('avg_order_value'),
        F.round(F.avg('actual_delivery_days'), 1) .alias('avg_delivery_days'),
    )
    .orderBy('review_score')
)

display(review_scores)


## 6. Gold Table 5 — Top Sellers by Revenue

In [ ]:
top_sellers = (
    silver_df
    .groupBy('seller_id', 'seller_city', 'seller_state')
    .agg(
        F.round(F.sum('payment_value'), 2).alias('total_revenue'),
        F.count('order_id')              .alias('total_orders'),
        F.round(F.avg('review_score'), 2) .alias('avg_review_score'),
    )
    .orderBy(F.desc('total_revenue'))
    .limit(100)
)

display(top_sellers)


## 7. Write All Gold Tables

In [ ]:
def write_gold(df, name):
    path = gold_base + name + '/'
    df.write.format('delta').mode('overwrite').save(path)
    print(f'Written: {path}')

write_gold(revenue_by_category,  'revenue_by_category')
write_gold(delivery_performance, 'delivery_performance')
write_gold(monthly_revenue,      'monthly_revenue_trend')
write_gold(review_scores,        'review_score_distribution')
write_gold(top_sellers,          'top_sellers')

print('\n✅  All Gold tables written successfully.')
